# 01  -  Setup

This notebook prepares the environment for the AI Functions cost demo. It is the first task in the bundle's job.

**What this notebook does:**
1. Reads job parameters (catalog, schema, owner email, etc.) from `dbutils.widgets`.
2. Creates the catalog and schema if they don't exist.
3. Validates that the running identity can read [`system.billing.usage`](https://docs.databricks.com/en/admin/system-tables/billing.html) and [`system.billing.list_prices`](https://docs.databricks.com/en/admin/system-tables/billing.html#schema-2-billing-list_prices) (needed for the reconciliation step at the end).
4. Prints the resolved configuration so re-runs are reproducible.

**Public references for this step:**
- [Unity Catalog: create catalogs and schemas](https://docs.databricks.com/en/data-governance/unity-catalog/create-catalogs.html)
- [Enable system tables](https://docs.databricks.com/en/admin/system-tables/index.html#enable-system-tables)
- [`dbutils.widgets` parameters in jobs](https://docs.databricks.com/en/jobs/create-run-jobs.html#configure-job-parameters)


## Read job parameters

The next cell pulls every config value from job parameters using `dbutils.widgets`. The bundle's `databricks.yml` declares these and the job passes them in. **No catalog or schema name is hardcoded anywhere in this notebook.**


In [ ]:
dbutils.widgets.text("catalog",      "main",                        "Unity Catalog catalog")
dbutils.widgets.text("schema",       "ai_functions_cost_demo",                "Schema (created if missing)")
dbutils.widgets.text("sample_size",  "100",                                   "Number of synthetic transcripts")
dbutils.widgets.text("team_tag",     "ai-cost-demo",                    "custom_tags.team value")
dbutils.widgets.text("owner_email",  "you@example.com",                       "custom_tags.Owner value")
dbutils.widgets.text("model_name",   "databricks-meta-llama-3-3-70b-instruct","Model endpoint for ai_query")

CATALOG     = dbutils.widgets.get("catalog")
SCHEMA      = dbutils.widgets.get("schema")
SAMPLE_SIZE = int(dbutils.widgets.get("sample_size"))
TEAM_TAG    = dbutils.widgets.get("team_tag")
OWNER_EMAIL = dbutils.widgets.get("owner_email")
MODEL_NAME  = dbutils.widgets.get("model_name")

print("Resolved configuration:")
print(f"  catalog     = {CATALOG}")
print(f"  schema      = {SCHEMA}")
print(f"  sample_size = {SAMPLE_SIZE}")
print(f"  team_tag    = {TEAM_TAG}")
print(f"  owner_email = {OWNER_EMAIL}")
print(f"  model_name  = {MODEL_NAME}")


## Create catalog and schema if missing

The next cell ensures the target catalog and schema exist before any downstream task tries to write a table. This is idempotent  -  re-running it on an existing catalog/schema is a no-op.

If the running identity does not have `CREATE CATALOG` permission, the catalog must already exist; only the schema will be created. See [UC privileges](https://docs.databricks.com/en/data-governance/unity-catalog/manage-privileges/privileges.html).


In [ ]:
from pyspark.sql import functions as F

# Step 1: check if the catalog already exists. This avoids a noisy stacktrace
# when a workspace has Default Storage disabled or no managed location set --
# in that case CREATE CATALOG fails even when the catalog already exists.
catalog_exists = (
    spark.sql("SHOW CATALOGS")
        .filter(F.col("catalog") == CATALOG)
        .count() > 0
)

if catalog_exists:
    print(f"Catalog ready (pre-existing): {CATALOG}")
else:
    # Step 2: catalog missing -> attempt to create. May still fail on
    # workspaces without Default Storage; surface a clean message if so.
    try:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
        print(f"Catalog created: {CATALOG}")
    except Exception as e:
        print(f"ERROR: catalog {CATALOG} does not exist and could not be auto-created.")
        print( "  Likely cause: Default Storage is not enabled and no managed location was provided.")
        print( "  Action: ask your UC admin to create the catalog, OR")
        print(f"          override the bundle variable so the demo uses a catalog you already have access to:")
        print(f"          databricks bundle deploy -t dev --var \"catalog=<your_existing_catalog>\"")
        raise

# Step 3: schema is always safe to create inside an existing catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Schema ready: {CATALOG}.{SCHEMA}")


## Validate `system.billing.usage` access

The final reconciliation step (notebook `04_reconcile_costs`) needs to query [`system.billing.usage`](https://docs.databricks.com/en/admin/system-tables/billing.html). The next cell runs a tiny test query and prints whether the running identity can see the table. If access is missing, the reconciliation step will skip gracefully  -  but it's better to know upfront.

If this check fails, your account admin needs to grant `USE SCHEMA` on `system.billing` and `SELECT` on `system.billing.usage`. See [enable system tables](https://docs.databricks.com/en/admin/system-tables/index.html#enable-system-tables).


In [ ]:
try:
    n = spark.sql("SELECT COUNT(*) AS n FROM system.billing.usage WHERE usage_date = CURRENT_DATE() - 1").collect()[0]["n"]
    print(f"system.billing.usage: ACCESSIBLE  (yesterday's row count: {n:,})")
except Exception as e:
    print("system.billing.usage: NOT ACCESSIBLE - reconciliation will be skipped.")
    print(f"  reason: {e}")


## Setup complete

Downstream notebooks will read `${dbutils.widgets.get('catalog')}.${dbutils.widgets.get('schema')}` for all tables. The job framework propagates the parameters automatically.

**Next:** `02_generate_data.ipynb`  -  synthetic call transcripts.
